# OVR Logistic Regression with Thresholding

This notebook builds a threshold-based one-vs-rest logistic regression benchmark for skill tagging using Supabase-hosted embeddings and labels.

What this notebook does:
- loads job, course, and skill embeddings from Supabase tables `job_embeddings`, `course_embeddings`, and `skill_embeddings`
- reads `extracted_skills` directly from the Supabase job and course tables
- builds two dataset variants: `jobs_only` and `jobs_plus_courses`
- trains one logistic regression model per skill on the train split
- tunes thresholds on the validation split only
- selects the final configuration using validation metrics
- reports the final locked metrics on the held-out test split

Before running the notebook, set `SUPABASE_URL` plus one of `SUPABASE_KEY`, `SUPABASE_SERVICE_ROLE_KEY`, or `SUPABASE_ANON_KEY` in your environment.


In [8]:
import ast
import numpy as np
import os
import pandas as pd
from pathlib import Path
import requests
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import train_test_split

ENV_PATH = Path("../.env")

def load_env_file(env_path):
    if not env_path.exists():
        return

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip(chr(34)).strip(chr(39))
        os.environ[key] = value

load_env_file(ENV_PATH)

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = (
    os.getenv("SUPABASE_KEY")
    or os.getenv("SUPABASE_SERVICE_ROLE_KEY")
    or os.getenv("SUPABASE_ANON_KEY")
)

if not SUPABASE_URL or "your_project_id" in SUPABASE_URL:
    raise ValueError(f"Invalid SUPABASE_URL loaded: {SUPABASE_URL!r}. Check ../.env and restart the kernel.")

PREVIEW_ONLY = False
PREVIEW_ROWS = 5



def parse_embedding(value):
    if isinstance(value, list):
        return [float(x) for x in value]
    if isinstance(value, str):
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [float(x) for x in parsed]
    raise TypeError(f"Unsupported embedding value type: {type(value)!r}")


def fetch_supabase_table(table_name, page_size=200, preview_only=False, preview_rows=5):
    if not SUPABASE_URL or not SUPABASE_KEY:
        raise ValueError(
            "Set SUPABASE_URL and one of SUPABASE_KEY, SUPABASE_SERVICE_ROLE_KEY, or SUPABASE_ANON_KEY before running this notebook."
        )

    base_url = f"{SUPABASE_URL.rstrip('/')}/rest/v1/{table_name}"
    headers = {
        "apikey": SUPABASE_KEY,
        "Authorization": f"Bearer {SUPABASE_KEY}",
        "Accept": "application/json",
    }

    if preview_only:
        print(f"Previewing {table_name} ({preview_rows} rows)...")
        response = requests.get(
            base_url,
            headers=headers,
            params={"select": "*", "limit": preview_rows},
            timeout=60,
        )
        if not response.ok:
            raise requests.HTTPError(
                f"Supabase preview failed for {table_name}: {response.status_code} {response.text}",
                response=response,
            )
        return pd.DataFrame(response.json())

    rows = []
    offset = 0
    while True:
        print(f"Fetching {table_name} rows {offset} to {offset + page_size - 1}...")
        response = requests.get(
            base_url,
            headers=headers,
            params={"select": "*", "limit": page_size, "offset": offset},
            timeout=60,
        )
        if not response.ok:
            raise requests.HTTPError(
                f"Supabase fetch failed for {table_name} at offset {offset}: {response.status_code} {response.text}",
                response=response,
            )

        batch = response.json()
        if not batch:
            break

        rows.extend(batch)
        if len(batch) < page_size:
            break
        offset += page_size

    return pd.DataFrame(rows)


def require_columns(df, required_columns, table_name):
    missing_columns = [column for column in required_columns if column not in df.columns]
    if missing_columns:
        raise KeyError(f"Table {table_name} is missing required columns: {missing_columns}")


skills_embeddings = fetch_supabase_table("skill_embeddings", preview_only=PREVIEW_ONLY, preview_rows=PREVIEW_ROWS)
jobs_embeddings = fetch_supabase_table("job_embeddings", preview_only=PREVIEW_ONLY, preview_rows=PREVIEW_ROWS)
courses_embeddings = fetch_supabase_table("course_embeddings", preview_only=PREVIEW_ONLY, preview_rows=PREVIEW_ROWS)

require_columns(skills_embeddings, ["skill_name", "embedding"], "skill_embeddings")
require_columns(jobs_embeddings, ["job_id", "title", "embedding", "extracted_skills"], "job_embeddings")
require_columns(courses_embeddings, ["course_code", "title", "embedding", "extracted_skills"], "course_embeddings")

skills_embeddings = skills_embeddings.copy()
if "record_id" not in skills_embeddings.columns:
    skills_embeddings["record_id"] = "skill:" + skills_embeddings["skill_name"].astype(str)
if "entity_type" not in skills_embeddings.columns:
    skills_embeddings["entity_type"] = "skill"
skills_embeddings["embedding"] = skills_embeddings["embedding"].apply(parse_embedding)

jobs_embeddings = jobs_embeddings.copy()
if "record_id" not in jobs_embeddings.columns:
    jobs_embeddings["record_id"] = "job:" + jobs_embeddings["job_id"].astype(str)
if "entity_type" not in jobs_embeddings.columns:
    jobs_embeddings["entity_type"] = "job"
jobs_embeddings["embedding"] = jobs_embeddings["embedding"].apply(parse_embedding)

courses_embeddings = courses_embeddings.copy()
if "record_id" not in courses_embeddings.columns:
    courses_embeddings["record_id"] = "course:" + courses_embeddings["course_code"].astype(str)
if "entity_type" not in courses_embeddings.columns:
    courses_embeddings["entity_type"] = "course"
courses_embeddings["embedding"] = courses_embeddings["embedding"].apply(parse_embedding)


print("Loaded preview data from Supabase:" if PREVIEW_ONLY else "Loaded embeddings from Supabase:")
print(f"skills rows:  {len(skills_embeddings)}")
print(f"jobs rows:    {len(jobs_embeddings)}")
print(f"courses rows: {len(courses_embeddings)}")
print()
print("jobs_embeddings head:")
print(jobs_embeddings[["record_id", "job_id", "title"]].head().to_string(index=False))
print()
print("courses_embeddings head:")
print(courses_embeddings[["record_id", "course_code", "title"]].head().to_string(index=False))
print()
print("skills_embeddings head:")
print(skills_embeddings[["record_id", "skill_name"]].head().to_string(index=False))
print()
print(f"num_skills: {len(skills_embeddings)}")

Fetching skill_embeddings rows 0 to 199...
Fetching skill_embeddings rows 200 to 399...
Fetching skill_embeddings rows 400 to 599...
Fetching skill_embeddings rows 600 to 799...
Fetching skill_embeddings rows 800 to 999...
Fetching skill_embeddings rows 1000 to 1199...
Fetching skill_embeddings rows 1200 to 1399...
Fetching skill_embeddings rows 1400 to 1599...
Fetching skill_embeddings rows 1600 to 1799...
Fetching skill_embeddings rows 1800 to 1999...
Fetching skill_embeddings rows 2000 to 2199...
Fetching skill_embeddings rows 2200 to 2399...
Fetching skill_embeddings rows 2400 to 2599...
Fetching skill_embeddings rows 2600 to 2799...
Fetching skill_embeddings rows 2800 to 2999...
Fetching skill_embeddings rows 3000 to 3199...
Fetching skill_embeddings rows 3200 to 3399...
Fetching skill_embeddings rows 3400 to 3599...
Fetching skill_embeddings rows 3600 to 3799...
Fetching skill_embeddings rows 3800 to 3999...
Fetching skill_embeddings rows 4000 to 4199...
Fetching skill_embeddings

## Helper Functions

These helpers do the repetitive setup work:
- convert `extracted_skills` text into Python lists
- join embeddings to labels
- build binary label matrices
- train one logistic regression per skill
- tune thresholds and compute metrics

Keeping them here makes the actual experiment cells shorter and easier to read.


In [9]:
def parse_skill_list(value):
    if pd.isna(value) or str(value).strip() == "":
        return []
    return [skill.strip() for skill in str(value).split("|") if skill.strip()]


def prepare_labeled_entity_frame(df, entity_id_col):
    prepared = df.copy()
    missing_mask = prepared["extracted_skills"].isna() | (prepared["extracted_skills"].astype(str).str.strip() == "")
    dropped_ids = prepared.loc[missing_mask, entity_id_col].tolist()
    if dropped_ids:
        print(f"Dropping {len(dropped_ids)} unlabeled rows for {entity_id_col}: {dropped_ids[:10]}" + (" ..." if len(dropped_ids) > 10 else ""))
        prepared = prepared.loc[~missing_mask].copy()

    prepared["actual_skill_lists"] = prepared["extracted_skills"].apply(parse_skill_list)
    prepared["display_title"] = prepared["title"]
    prepared["entity_id"] = prepared[entity_id_col]

    return prepared[["entity_type", "entity_id", "display_title", "embedding", "actual_skill_lists"]].copy()


def split_entity_frame(df, random_state=42):
    train_df, temp_df = train_test_split(df, test_size=0.4, random_state=random_state, shuffle=True)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=random_state, shuffle=True)
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


def build_indicator_matrix(skill_lists, skill_names):
    skill_to_idx = {skill: idx for idx, skill in enumerate(skill_names)}
    indicator = np.zeros((len(skill_lists), len(skill_names)), dtype=np.uint8)

    for row_idx, skills in enumerate(skill_lists):
        for skill in skills:
            if skill in skill_to_idx:
                indicator[row_idx, skill_to_idx[skill]] = 1

    return indicator


def build_dataset(train_df, val_df, test_df, skill_names):
    X_train = np.vstack(train_df["embedding"].to_numpy()).astype(np.float32)
    X_val = np.vstack(val_df["embedding"].to_numpy()).astype(np.float32)
    X_test = np.vstack(test_df["embedding"].to_numpy()).astype(np.float32)

    Y_train = build_indicator_matrix(train_df["actual_skill_lists"].tolist(), skill_names)
    Y_val = build_indicator_matrix(val_df["actual_skill_lists"].tolist(), skill_names)
    Y_test = build_indicator_matrix(test_df["actual_skill_lists"].tolist(), skill_names)

    return X_train, X_val, X_test, Y_train, Y_val, Y_test


def fit_ovr_probabilities(X_train, Y_train, X_val, X_test, skill_names):
    val_proba = np.zeros((X_val.shape[0], len(skill_names)), dtype=np.float32)
    test_proba = np.zeros((X_test.shape[0], len(skill_names)), dtype=np.float32)
    fitted_mask = np.zeros(len(skill_names), dtype=bool)
    skipped_skills = []

    for j, skill_name in enumerate(skill_names):
        y_train_j = Y_train[:, j]

        if len(np.unique(y_train_j)) < 2:
            skipped_skills.append(skill_name)
            continue

        model = LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42,
        )
        model.fit(X_train, y_train_j)

        val_proba[:, j] = model.predict_proba(X_val)[:, 1]
        test_proba[:, j] = model.predict_proba(X_test)[:, 1]
        fitted_mask[j] = True

    return val_proba, test_proba, fitted_mask, skipped_skills


def micro_metrics(y_true, y_pred):
    tp = np.logical_and(y_pred == 1, y_true == 1).sum()
    fp = np.logical_and(y_pred == 1, y_true == 0).sum()
    fn = np.logical_and(y_pred == 0, y_true == 1).sum()

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

    return precision, recall, f1


def macro_metrics(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    return precision, recall, f1


def find_best_threshold(y_true, y_score):
    flat_true = np.asarray(y_true, dtype=np.uint8).ravel()
    flat_score = np.asarray(y_score, dtype=np.float32).ravel()

    if flat_score.size == 0:
        return 0.5

    order = np.argsort(flat_score, kind="mergesort")[::-1]
    sorted_score = flat_score[order]
    sorted_true = flat_true[order]

    tp_cum = np.cumsum(sorted_true, dtype=np.int64)
    fp_cum = np.cumsum(1 - sorted_true, dtype=np.int64)
    total_positive = int(sorted_true.sum())

    distinct_mask = np.r_[sorted_score[1:] != sorted_score[:-1], True]
    distinct_end_idx = np.flatnonzero(distinct_mask)

    tp = tp_cum[distinct_end_idx]
    fp = fp_cum[distinct_end_idx]
    fn = total_positive - tp

    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=np.float64), where=(tp + fp) != 0)
    recall = np.divide(tp, tp + fn, out=np.zeros_like(tp, dtype=np.float64), where=(tp + fn) != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(precision), where=(precision + recall) != 0)
    thresholds = sorted_score[distinct_end_idx].astype(np.float64)

    best_idx = 0
    best_candidate = (float(f1[0]), float(recall[0]), -float(thresholds[0]))
    for idx in range(1, len(thresholds)):
        candidate = (float(f1[idx]), float(recall[idx]), -float(thresholds[idx]))
        if candidate > best_candidate:
            best_candidate = candidate
            best_idx = idx

    none_positive_candidate = (0.0, 0.0, -(1.0 + 1e-9))
    if none_positive_candidate > best_candidate:
        return float(1.0 + 1e-9)

    return float(thresholds[best_idx])


def tune_global_threshold(y_true, y_score):
    return find_best_threshold(y_true, y_score)


def tune_best_threshold_for_one_skill(y_true, y_score):
    return find_best_threshold(y_true, y_score)


def indicator_row_to_skills(row, skill_names):
    return [skill_names[j] for j, flag in enumerate(row) if flag == 1]


## Prepare Supabase Labels

At this point we use the `extracted_skills` already stored in Supabase on the job and course tables.

This turns the raw embedding rows into supervised learning data:
- the embedding column gives the feature vector
- the `extracted_skills` column gives the true skill list

After preparation, each row has:
- entity id
- title
- embedding
- actual skill list


In [10]:
skill_names = skills_embeddings["skill_name"].tolist()

jobs_all_df = prepare_labeled_entity_frame(jobs_embeddings.copy(), entity_id_col="job_id")
courses_all_df = prepare_labeled_entity_frame(courses_embeddings.copy(), entity_id_col="course_code")

jobs_train_df, jobs_val_df, jobs_test_df = split_entity_frame(jobs_all_df, random_state=42)
courses_train_df, courses_val_df, courses_test_df = split_entity_frame(courses_all_df, random_state=42)

print("Supervised rows after reading Supabase labels and in-notebook splitting:")
print(f"jobs train:    {len(jobs_train_df)}")
print(f"jobs val:      {len(jobs_val_df)}")
print(f"jobs test:     {len(jobs_test_df)}")
print(f"courses train: {len(courses_train_df)}")
print(f"courses val:   {len(courses_val_df)}")
print(f"courses test:  {len(courses_test_df)}")


Dropping 1 unlabeled rows for job_id: ['30d38911187b8c345cd04c7b000062e1']
Dropping 1311 unlabeled rows for course_code: ['ACC3712', 'ACC3713', 'BSE3761', 'BSE4761', 'BSE4761X', 'CM2111', 'CM2121', 'CM3232', 'CM3291', 'CM3295'] ...
Supervised rows after reading Supabase labels and in-notebook splitting:
jobs train:    951
jobs val:      317
jobs test:     318
courses train: 2495
courses val:   832
courses test:  832


## Build Benchmark Variants

We evaluate two scopes:
- `jobs_only`: only job postings are used
- `jobs_plus_courses`: job postings and ACC courses are combined

This helps us compare the OVR model against the same kinds of setups explored in the cosine notebook.


In [11]:
dataset_variants = {
    "jobs_only": {
        "train": jobs_train_df,
        "val": jobs_val_df,
        "test": jobs_test_df,
    },
    "jobs_plus_courses": {
        "train": pd.concat([jobs_train_df, courses_train_df], ignore_index=True),
        "val": pd.concat([jobs_val_df, courses_val_df], ignore_index=True),
        "test": pd.concat([jobs_test_df, courses_test_df], ignore_index=True),
    },
}

for variant_name, frames in dataset_variants.items():
    print(variant_name)
    print(f"  train rows: {len(frames['train'])}")
    print(f"  val rows:   {len(frames['val'])}")
    print(f"  test rows:  {len(frames['test'])}")


jobs_only
  train rows: 951
  val rows:   317
  test rows:  318
jobs_plus_courses
  train rows: 3446
  val rows:   1149
  test rows:  1150


## Train, Tune, and Evaluate

This is the main experiment cell.

For each dataset variant, it:
- builds feature and label matrices
- trains one logistic regression model per skill
- tunes threshold(s) on validation
- evaluates the locked predictions on test

It stores both a summary table and prediction previews so we can inspect what the model is actually predicting.


In [12]:
results = []
artifacts = {}

for variant_name, frames in dataset_variants.items():
    X_train, X_val, X_test, Y_train, Y_val, Y_test = build_dataset(
        frames["train"],
        frames["val"],
        frames["test"],
        skill_names,
    )

    val_proba, test_proba, fitted_mask, skipped_skills = fit_ovr_probabilities(
        X_train,
        Y_train,
        X_val,
        X_test,
        skill_names,
    )

    global_threshold = tune_global_threshold(Y_val, val_proba)
    global_val_pred = (val_proba >= global_threshold).astype(np.uint8)
    global_val_micro_precision, global_val_micro_recall, global_val_micro_f1 = micro_metrics(Y_val, global_val_pred)
    global_val_macro_precision, global_val_macro_recall, global_val_macro_f1 = macro_metrics(Y_val, global_val_pred)
    global_test_pred = (test_proba >= global_threshold).astype(np.uint8)
    global_test_micro_precision, global_test_micro_recall, global_test_micro_f1 = micro_metrics(Y_test, global_test_pred)
    global_test_macro_precision, global_test_macro_recall, global_test_macro_f1 = macro_metrics(Y_test, global_test_pred)

    results.append({
        "dataset_variant": variant_name,
        "threshold_type": "global",
        "val_micro_precision": global_val_micro_precision,
        "val_micro_recall": global_val_micro_recall,
        "val_micro_f1": global_val_micro_f1,
        "val_macro_precision": global_val_macro_precision,
        "val_macro_recall": global_val_macro_recall,
        "val_macro_f1": global_val_macro_f1,
        "test_micro_precision": global_test_micro_precision,
        "test_micro_recall": global_test_micro_recall,
        "test_micro_f1": global_test_micro_f1,
        "test_macro_precision": global_test_macro_precision,
        "test_macro_recall": global_test_macro_recall,
        "test_macro_f1": global_test_macro_f1,
        "global_threshold": global_threshold,
        "fitted_models": int(fitted_mask.sum()),
        "skipped_skills": len(skipped_skills),
        "train_rows": len(frames["train"]),
        "val_rows": len(frames["val"]),
        "test_rows": len(frames["test"]),
    })

    global_predicted_skill_lists = [indicator_row_to_skills(row, skill_names) for row in global_test_pred]
    artifacts[(variant_name, "global")] = {
        "predictions_df": pd.DataFrame({
            "entity_type": frames["test"]["entity_type"],
            "entity_id": frames["test"]["entity_id"],
            "title": frames["test"]["display_title"],
            "actual_skills": [" | ".join(skills) for skills in frames["test"]["actual_skill_lists"]],
            "predicted_skills": [" | ".join(skills) for skills in global_predicted_skill_lists],
            "tp": [len(set(a) & set(p)) for a, p in zip(frames["test"]["actual_skill_lists"], global_predicted_skill_lists)],
            "fp": [len(set(p) - set(a)) for a, p in zip(frames["test"]["actual_skill_lists"], global_predicted_skill_lists)],
            "fn": [len(set(a) - set(p)) for a, p in zip(frames["test"]["actual_skill_lists"], global_predicted_skill_lists)],
        }),
        "thresholds_df": pd.DataFrame({
            "skill_name": skill_names,
            "threshold": np.full(len(skill_names), global_threshold, dtype=np.float32),
            "was_fitted": fitted_mask,
            "train_positive_support": Y_train.sum(axis=0),
            "val_positive_support": Y_val.sum(axis=0),
            "test_positive_support": Y_test.sum(axis=0),
        }),
    }

    skill_thresholds = np.full(len(skill_names), 0.5, dtype=np.float32)
    for j in np.where(fitted_mask)[0]:
        skill_thresholds[j] = tune_best_threshold_for_one_skill(Y_val[:, j], val_proba[:, j])

    skill_val_pred = (val_proba >= skill_thresholds[np.newaxis, :]).astype(np.uint8)
    skill_val_micro_precision, skill_val_micro_recall, skill_val_micro_f1 = micro_metrics(Y_val, skill_val_pred)
    skill_val_macro_precision, skill_val_macro_recall, skill_val_macro_f1 = macro_metrics(Y_val, skill_val_pred)
    skill_test_pred = (test_proba >= skill_thresholds[np.newaxis, :]).astype(np.uint8)
    skill_test_micro_precision, skill_test_micro_recall, skill_test_micro_f1 = micro_metrics(Y_test, skill_test_pred)
    skill_test_macro_precision, skill_test_macro_recall, skill_test_macro_f1 = macro_metrics(Y_test, skill_test_pred)

    results.append({
        "dataset_variant": variant_name,
        "threshold_type": "skill_specific",
        "val_micro_precision": skill_val_micro_precision,
        "val_micro_recall": skill_val_micro_recall,
        "val_micro_f1": skill_val_micro_f1,
        "val_macro_precision": skill_val_macro_precision,
        "val_macro_recall": skill_val_macro_recall,
        "val_macro_f1": skill_val_macro_f1,
        "test_micro_precision": skill_test_micro_precision,
        "test_micro_recall": skill_test_micro_recall,
        "test_micro_f1": skill_test_micro_f1,
        "test_macro_precision": skill_test_macro_precision,
        "test_macro_recall": skill_test_macro_recall,
        "test_macro_f1": skill_test_macro_f1,
        "global_threshold": np.nan,
        "fitted_models": int(fitted_mask.sum()),
        "skipped_skills": len(skipped_skills),
        "train_rows": len(frames["train"]),
        "val_rows": len(frames["val"]),
        "test_rows": len(frames["test"]),
    })

    skill_predicted_skill_lists = [indicator_row_to_skills(row, skill_names) for row in skill_test_pred]
    artifacts[(variant_name, "skill_specific")] = {
        "predictions_df": pd.DataFrame({
            "entity_type": frames["test"]["entity_type"],
            "entity_id": frames["test"]["entity_id"],
            "title": frames["test"]["display_title"],
            "actual_skills": [" | ".join(skills) for skills in frames["test"]["actual_skill_lists"]],
            "predicted_skills": [" | ".join(skills) for skills in skill_predicted_skill_lists],
            "tp": [len(set(a) & set(p)) for a, p in zip(frames["test"]["actual_skill_lists"], skill_predicted_skill_lists)],
            "fp": [len(set(p) - set(a)) for a, p in zip(frames["test"]["actual_skill_lists"], skill_predicted_skill_lists)],
            "fn": [len(set(a) - set(p)) for a, p in zip(frames["test"]["actual_skill_lists"], skill_predicted_skill_lists)],
        }),
        "thresholds_df": pd.DataFrame({
            "skill_name": skill_names,
            "threshold": skill_thresholds,
            "was_fitted": fitted_mask,
            "train_positive_support": Y_train.sum(axis=0),
            "val_positive_support": Y_val.sum(axis=0),
            "test_positive_support": Y_test.sum(axis=0),
        }),
    }

comparison_df = pd.DataFrame(results).sort_values(
    by=["val_micro_f1", "val_micro_recall", "val_macro_f1"],
    ascending=[False, False, False],
).reset_index(drop=True)
comparison_df


,dataset_variant,threshold_type,val_micro_precision,val_micro_recall,val_micro_f1,val_macro_precision,val_macro_recall,val_macro_f1,test_micro_precision,test_micro_recall,test_micro_f1,test_macro_precision,test_macro_recall,test_macro_f1,global_threshold,fitted_models,skipped_skills,train_rows,val_rows,test_rows
0,jobs_plus_courses,global,0.502111,0.511114,0.506573,0.106648,0.106484,0.098205,0.485781,0.473343,0.479481,0.096423,0.094127,0.088060,0.822794,1423,2978,3446,1149,1150
1,jobs_only,global,0.532132,0.450592,0.487979,0.084759,0.082206,0.077142,0.504023,0.416133,0.455880,0.077739,0.071981,0.068816,0.720205,1355,3046,951,317,318
2,jobs_only,skill_specific,0.018042,0.658015,0.035121,0.120820,0.159753,0.114468,0.013058,0.493476,0.025443,0.062325,0.109299,0.054386,NaN,1355,3046,951,317,318
3,jobs_plus_courses,skill_specific,0.010332,0.646858,0.020339,0.147688,0.181871,0.135565,0.007909,0.501657,0.015573,0.083870,0.125310,0.071116,NaN,1423,2978,3446,1149,1150


## Cache Experiment Artifacts

The experiment cell above is the expensive part of this notebook.

After it finishes, run the save cell below once to cache `comparison_df` and `artifacts` under `../data/cache/notebooks/`.

If the kernel restarts later and you only want to continue with the downstream analysis cells, skip the expensive experiment cell and run the load cell instead.


In [13]:
import pickle

CACHE_DIR = Path("../data/cache/notebooks")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = CACHE_DIR / "ovr_thres_fixed_all_experiment_cache_v2.pkl"

cache_payload = {
    "comparison_df": comparison_df,
    "artifacts": artifacts,
}

with CACHE_PATH.open("wb") as f:
    pickle.dump(cache_payload, f)

print(f"Saved experiment cache to: {CACHE_PATH.resolve()}")


Saved experiment cache to: C:\Users\ryant\Desktop\Data_Science\Projects\dsa4264-JobReady\data\cache\notebooks\ovr_thres_fixed_all_experiment_cache_v2.pkl


In [14]:
import pickle

CACHE_DIR = Path("../data/cache/notebooks")
CACHE_PATH = CACHE_DIR / "ovr_thres_fixed_all_experiment_cache_v2.pkl"

with CACHE_PATH.open("rb") as f:
    cache_payload = pickle.load(f)

comparison_df = cache_payload["comparison_df"]
artifacts = cache_payload["artifacts"]

print(f"Loaded experiment cache from: {CACHE_PATH.resolve()}")
display(comparison_df)


Loaded experiment cache from: C:\Users\ryant\Desktop\Data_Science\Projects\dsa4264-JobReady\data\cache\notebooks\ovr_thres_fixed_all_experiment_cache_v2.pkl


,dataset_variant,threshold_type,val_micro_precision,val_micro_recall,val_micro_f1,val_macro_precision,val_macro_recall,val_macro_f1,test_micro_precision,test_micro_recall,test_micro_f1,test_macro_precision,test_macro_recall,test_macro_f1,global_threshold,fitted_models,skipped_skills,train_rows,val_rows,test_rows
0,jobs_plus_courses,global,0.502111,0.511114,0.506573,0.106648,0.106484,0.098205,0.485781,0.473343,0.479481,0.096423,0.094127,0.088060,0.822794,1423,2978,3446,1149,1150
1,jobs_only,global,0.532132,0.450592,0.487979,0.084759,0.082206,0.077142,0.504023,0.416133,0.455880,0.077739,0.071981,0.068816,0.720205,1355,3046,951,317,318
2,jobs_only,skill_specific,0.018042,0.658015,0.035121,0.120820,0.159753,0.114468,0.013058,0.493476,0.025443,0.062325,0.109299,0.054386,NaN,1355,3046,951,317,318
3,jobs_plus_courses,skill_specific,0.010332,0.646858,0.020339,0.147688,0.181871,0.135565,0.007909,0.501657,0.015573,0.083870,0.125310,0.071116,NaN,1423,2978,3446,1149,1150


## Inspect the Best Configuration

This final code cell selects the strongest validation result from the comparison table and then shows the locked test metrics plus a small prediction preview.

That makes it easier to check whether the model is making sensible skill predictions instead of relying on summary metrics alone.


In [15]:
best_row = comparison_df.iloc[0]
best_key = (best_row["dataset_variant"], best_row["threshold_type"])

selection_columns = [
    "dataset_variant",
    "threshold_type",
    "val_micro_precision",
    "val_micro_recall",
    "val_micro_f1",
    "val_macro_precision",
    "val_macro_recall",
    "val_macro_f1",
    "global_threshold",
    "fitted_models",
    "skipped_skills",
    "train_rows",
    "val_rows",
    "test_rows",
]
test_metric_columns = [
    "test_micro_precision",
    "test_micro_recall",
    "test_micro_f1",
    "test_macro_precision",
    "test_macro_recall",
    "test_macro_f1",
]

print("Selected configuration based on validation performance:")
print(best_row[selection_columns].to_string())
print()
print("Locked test metrics for the selected configuration:")
print(best_row[test_metric_columns].to_string())
print()
print("Test-set prediction preview:")
print(artifacts[best_key]["predictions_df"].head(10).to_string(index=False))


Selected configuration based on validation performance:
dataset_variant        jobs_plus_courses
threshold_type                    global
val_micro_precision             0.502111
val_micro_recall                0.511114
val_micro_f1                    0.506573
val_macro_precision             0.106648
val_macro_recall                0.106484
val_macro_f1                    0.098205
global_threshold                0.822794
fitted_models                       1423
skipped_skills                      2978
train_rows                          3446
val_rows                            1149
test_rows                           1150

Locked test metrics for the selected configuration:
test_micro_precision    0.485781
test_micro_recall       0.473343
test_micro_f1           0.479481
test_macro_precision    0.096423
test_macro_recall       0.094127
test_macro_f1            0.08806

Test-set prediction preview:
entity_type                        entity_id                                             

## Final Skills Gap Table

This section refits the selected final model on the full labeled dataset (`train + val + test`) and scores all labeled jobs and courses.

It then creates a high-demand skills table using:
- predicted job demand across all jobs
- predicted curriculum coverage across all courses
- a rule-based gap interpretation from those two signals


In [16]:
HIGH_DEMAND_THRESHOLD_PERCENT = 10.0
STRONG_COURSE_COVERAGE_THRESHOLD_PERCENT = 10.0

best_variant_name = str(best_row["dataset_variant"])
best_threshold_type = str(best_row["threshold_type"])
best_artifact = artifacts[best_key]

if best_variant_name == "jobs_only":
    full_fit_df = jobs_all_df.reset_index(drop=True).copy()
else:
    full_fit_df = pd.concat([jobs_all_df, courses_all_df], ignore_index=True)

X_full_fit = np.vstack(full_fit_df["embedding"].to_numpy()).astype(np.float32)
Y_full_fit = build_indicator_matrix(full_fit_df["actual_skill_lists"].tolist(), skill_names)
X_jobs_all = np.vstack(jobs_all_df["embedding"].to_numpy()).astype(np.float32)
X_courses_all = np.vstack(courses_all_df["embedding"].to_numpy()).astype(np.float32)

jobs_full_proba = np.zeros((X_jobs_all.shape[0], len(skill_names)), dtype=np.float32)
courses_full_proba = np.zeros((X_courses_all.shape[0], len(skill_names)), dtype=np.float32)
full_fitted_mask = np.zeros(len(skill_names), dtype=bool)
full_skipped_skills = []

for j, skill_name in enumerate(skill_names):
    y_full_j = Y_full_fit[:, j]

    if len(np.unique(y_full_j)) < 2:
        full_skipped_skills.append(skill_name)
        continue

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42,
    )
    model.fit(X_full_fit, y_full_j)

    jobs_full_proba[:, j] = model.predict_proba(X_jobs_all)[:, 1]
    courses_full_proba[:, j] = model.predict_proba(X_courses_all)[:, 1]
    full_fitted_mask[j] = True

if best_threshold_type == "global":
    selected_thresholds = np.full(len(skill_names), float(best_row["global_threshold"]), dtype=np.float32)
else:
    selected_thresholds = best_artifact["thresholds_df"]["threshold"].to_numpy(dtype=np.float32)

jobs_full_pred = (jobs_full_proba >= selected_thresholds[np.newaxis, :]).astype(np.uint8)
courses_full_pred = (courses_full_proba >= selected_thresholds[np.newaxis, :]).astype(np.uint8)

job_skill_counts = jobs_full_pred.sum(axis=0).astype(int)
course_skill_counts = courses_full_pred.sum(axis=0).astype(int)
job_denominator = max(len(jobs_all_df), 1)
course_denominator = max(len(courses_all_df), 1)

gap_rows = []
for j, skill_name in enumerate(skill_names):
    job_demand_percent = 100.0 * job_skill_counts[j] / job_denominator
    course_coverage_percent = 100.0 * course_skill_counts[j] / course_denominator
    job_demand = "High" if job_demand_percent >= HIGH_DEMAND_THRESHOLD_PERCENT else "Low"
    taught_in_any_course = course_skill_counts[j] > 0

    if course_coverage_percent >= STRONG_COURSE_COVERAGE_THRESHOLD_PERCENT:
        curriculum_coverage = "Strong"
    elif taught_in_any_course:
        curriculum_coverage = "Weak"
    else:
        curriculum_coverage = "Very weak"

    if job_demand == "High" and curriculum_coverage == "Strong":
        gap_interpretation = "Well covered"
    elif job_demand == "High" and curriculum_coverage == "Weak":
        gap_interpretation = "Needs strengthening"
    elif job_demand == "High":
        gap_interpretation = "Important gap - High demand skill with no detected course coverage"
    elif curriculum_coverage == "Strong":
        gap_interpretation = "Coverage exceeds current demand"
    elif curriculum_coverage == "Weak":
        gap_interpretation = "Low demand skill with some curriculum coverage"
    else:
        gap_interpretation = "Low demand and limited coverage"

    gap_rows.append({
        "skill_name": skill_name,
        "job_demand_percent": float(job_demand_percent),
        "job_demand": job_demand,
        "taught_in_any_course": "Yes" if taught_in_any_course else "No",
        "courses_covering_skill": int(course_skill_counts[j]),
        "course_coverage_percent": float(course_coverage_percent),
        "curriculum_coverage": curriculum_coverage,
        "gap_interpretation": gap_interpretation,
    })

full_model_gap_table_df = pd.DataFrame(gap_rows).sort_values(
    by=["job_demand", "job_demand_percent", "course_coverage_percent", "skill_name"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

high_demand_gap_table_df = full_model_gap_table_df.loc[
    full_model_gap_table_df["job_demand"] == "High"
].reset_index(drop=True)

final_model_refit_summary_df = pd.DataFrame({
    "selected_dataset_variant": [best_variant_name],
    "selected_threshold_type": [best_threshold_type],
    "full_fit_rows": [len(full_fit_df)],
    "jobs_scored": [len(jobs_all_df)],
    "courses_scored": [len(courses_all_df)],
    "fitted_skills_after_refit": [int(full_fitted_mask.sum())],
    "skipped_skills_after_refit": [len(full_skipped_skills)],
    "high_demand_threshold_percent": [HIGH_DEMAND_THRESHOLD_PERCENT],
    "strong_course_coverage_threshold_percent": [STRONG_COURSE_COVERAGE_THRESHOLD_PERCENT],
})

print("Final model refit summary:")
display(final_model_refit_summary_df)
print()
print("High-demand skills gap table:")
display(high_demand_gap_table_df[[
    "skill_name",
    "job_demand_percent",
    "job_demand",
    "taught_in_any_course",
    "courses_covering_skill",
    "course_coverage_percent",
    "curriculum_coverage",
    "gap_interpretation",
]])


Final model refit summary:


,selected_dataset_variant,selected_threshold_type,full_fit_rows,jobs_scored,courses_scored,fitted_skills_after_refit,skipped_skills_after_refit,high_demand_threshold_percent,strong_course_coverage_threshold_percent
0,jobs_plus_courses,global,5745,1586,4159,1522,2879,10.0,10.0



High-demand skills gap table:


,skill_name,job_demand_percent,job_demand,taught_in_any_course,courses_covering_skill,course_coverage_percent,curriculum_coverage,gap_interpretation
0,Business Needs Analysis,23.707440,High,Yes,44,1.057947,Weak,Needs strengthening
1,Infrastructure Support,19.230769,High,No,0,0.000000,Very weak,Important gap - High demand skill with no dete...
2,Applications Development,18.284994,High,Yes,194,4.664583,Weak,Needs strengthening
3,Programme Delivery,17.465322,High,No,0,0.000000,Very weak,Important gap - High demand skill with no dete...
4,Account Management,17.023960,High,No,0,0.000000,Very weak,Important gap - High demand skill with no dete...
5,Staff Communication and Engagement,15.510719,High,No,0,0.000000,Very weak,Important gap - High demand skill with no dete...
6,Job Analysis and Evaluation,12.295082,High,Yes,29,0.697283,Weak,Needs strengthening
7,Effective Client Communication,11.979823,High,Yes,2,0.048088,Weak,Needs strengthening
8,Construction Technology,11.538462,High,Yes,19,0.456841,Weak,Needs strengthening
9,Engineering Problem Solving,11.286255,High,Yes,57,1.370522,Weak,Needs strengthening


## Summary

This notebook now reads top to bottom as one clean benchmark:
- load data
- prepare supervised labels
- define the experiment variants
- train the OVR models on the train split
- tune thresholds and choose the final configuration on validation
- report the locked test result only for the selected configuration
- refit the selected final model on the full labeled dataset to build a high-demand skills gap table

That keeps the held-out test split as the final unbiased check instead of using it to decide which variant wins.

### Caveats and Limitations
- This benchmark assumes the CSV `extracted_skills` labels are the ground truth.
- Many skills are rare, so the model learns much better for common skills than rare ones.
- OVR cannot train a classifier for a skill if the train split does not contain both positive and negative examples for that skill.
- Micro-F1 is driven more by common skills, so it can look stronger than macro-F1.
- The skill-specific-threshold variant is more flexible, but it can overfit when the validation set is small.
- One course, `ACC4761E`, is excluded because it currently has no labeled `extracted_skills` value.
